# Clase 219 — Híbridos: weighted, switching, LightFM

3 estrategias hybrid sobre un dataset sintético + cold-start eval.

Requiere: `pip install lightfm scipy scikit-learn`.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(42)
n_users, n_items, n_features = 500, 200, 10

# Item features (géneros multi-hot) y user preferences latentes
item_features = rng.binomial(1, 0.25, (n_items, n_features)).astype(float)
user_prefs = rng.normal(0, 1, (n_users, n_features))

# Generar ratings sintéticos: user gusta items que matchean sus preferencias
scores_true = user_prefs @ item_features.T
noise = rng.normal(0, 0.5, scores_true.shape)
interact_prob = 1 / (1 + np.exp(-(scores_true + noise)))
R = (rng.random(scores_true.shape) < 0.1 * interact_prob).astype(float)
R_sparse = csr_matrix(R)
print(f'interactions: {R_sparse.nnz:,}')

## 1. Scores CF (sim cosine entre users) + scores content

In [ ]:
# CF: user-based prediction
sim_users = cosine_similarity(R_sparse)
np.fill_diagonal(sim_users, 0)
scores_cf = sim_users @ R   # (n_users, n_items)

# Content: user_profile = mean de item_features de items vistos
user_profiles_cb = (R @ item_features) / np.maximum(R.sum(axis=1, keepdims=True), 1)
scores_cb = user_profiles_cb @ item_features.T

# Normalizar a [0, 1] por user para mezclar
def minmax_per_user(s):
    mn = s.min(axis=1, keepdims=True)
    mx = s.max(axis=1, keepdims=True)
    return (s - mn) / np.maximum(mx - mn, 1e-9)

scores_cf_n = minmax_per_user(scores_cf)
scores_cb_n = minmax_per_user(scores_cb)
print('scores normalized — listos para mezclar.')

## 2. Weighted hybrid: tunear `α`

In [ ]:
def recall_at_k(scores, R_train, R_test, k=10):
    """Recall@k promedio. Excluye items en train."""
    s = scores.copy()
    s[R_train > 0] = -1
    top_k = np.argsort(-s, axis=1)[:, :k]
    hits = np.array([(R_test[u, top_k[u]] > 0).sum() / max((R_test[u] > 0).sum(), 1) for u in range(s.shape[0])])
    return hits.mean()

# Split train/test: 80/20 random por interaction
test_mask = (rng.random(R.shape) < 0.2) & (R > 0)
R_train = R.copy(); R_train[test_mask] = 0
R_test = np.where(test_mask, R, 0)

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
print(f'{"alpha":>6}  recall@10')
for a in alphas:
    s = a * scores_cf_n + (1 - a) * scores_cb_n
    r = recall_at_k(s, R_train, R_test, k=10)
    print(f'{a:>6.2f}  {r:.4f}')

## 3. Switching: content para cold-start, CF para mature

In [ ]:
interactions_per_user = (R_train > 0).sum(axis=1)
cold = interactions_per_user < 5
print(f'cold users (<5 interactions train): {cold.sum()} / {n_users}')

scores_switching = np.where(cold[:, None], scores_cb_n, scores_cf_n)
r_switch = recall_at_k(scores_switching, R_train, R_test)

# Separar evaluación por segmento
for seg_name, mask_seg in [('cold (<5)', cold), ('mature (>=5)', ~cold)]:
    for name, s in [('CF only', scores_cf_n), ('CB only', scores_cb_n), ('switching', scores_switching)]:
        r = recall_at_k(s[mask_seg], R_train[mask_seg], R_test[mask_seg])
        print(f'  segment={seg_name:14} model={name:12}  recall@10={r:.4f}')

## 4. LightFM hybrid

In [ ]:
try:
    from lightfm import LightFM
    from lightfm.evaluation import precision_at_k, recall_at_k as lfm_recall

    item_features_sp = csr_matrix(np.hstack([np.eye(n_items), item_features]))   # identity + features

    model = LightFM(loss='warp', no_components=32, random_state=42)
    model.fit(csr_matrix(R_train), item_features=item_features_sp, epochs=20, num_threads=2)

    r_lfm = lfm_recall(model, csr_matrix(R_test), train_interactions=csr_matrix(R_train),
                       item_features=item_features_sp, k=10, num_threads=2).mean()
    print(f'LightFM hybrid recall@10: {r_lfm:.4f}')
except ImportError:
    print('pip install lightfm para esta celda')

## 5. Tabla comparativa

In [ ]:
import pandas as pd
summary = pd.DataFrame([
    {'model': 'CF (CF only)',                  'recall@10': recall_at_k(scores_cf_n, R_train, R_test)},
    {'model': 'Content (CB only)',             'recall@10': recall_at_k(scores_cb_n, R_train, R_test)},
    {'model': 'Weighted (a=0.5)',              'recall@10': recall_at_k(0.5 * scores_cf_n + 0.5 * scores_cb_n, R_train, R_test)},
    {'model': 'Switching (cold→CB, else CF)',  'recall@10': r_switch},
]).round(4)
print(summary.to_string(index=False))

## Ejercicio guiado

1. Replicá sobre MovieLens 100K real. Mostrá `α` óptimo y NDCG@10 por segmento.
2. Implementá **cascade**: top-50 con CF → re-rank top-10 con content. ¿Mejora?
3. **Mixed carousel**: para una página, mostrar 5 items de CF + 5 de content + 5 de popularidad. Discutí trade-off engagement vs diversity.
4. Cold-start total: agregá user_id=999 sin interactions. Mostrá qué le recomienda cada modelo (CF: nada; content+demographics: razonable; LightFM con user features: aún mejor).
5. Bonus: tuneo continuo de `α` con bandit (epsilon-greedy o Thompson sampling).

## Conclusiones

- Weighted hybrid: simple, efectivo, fácil de tunear.
- Switching: ideal para cold-start, requiere definir umbral.
- LightFM aprende un modelo único con CF + features — mejor calidad con menos engineering.
- En producción: mixed carousels son lo más común (Spotify, Netflix).